# gpt-oss-20b 量化全流程

这个 notebook 说明 gpt-oss-20b 的量化和压缩路线。重点是 MXFP4，同时也解释 bitsandbytes、GGUF、AWQ、GPTQ、EXL2、FP8 在 gpt-oss 场景里的位置。

## 1. 排名和选择

| 排名 | 方法 | gpt-oss 场景 |
| --- | --- | --- |
| 1 | MXFP4 / 官方压缩权重 | gpt-oss 的核心路线 |
| 2 | Ollama 官方模型 | 本地运行最简单 |
| 3 | vLLM 支持的量化 | 服务端部署 |
| 4 | bitsandbytes 4bit | QLoRA 训练加载 |
| 5 | GGUF | llama.cpp/Ollama 自定义路线 |
| 6 | AWQ/GPTQ/FP8/EXL2 | 取决于框架版本和社区支持 |

## 2. MXFP4

MXFP4 是 gpt-oss 权重压缩重点。它不是 QLoRA 的 4bit，不要混为一谈。

理解方式：

- bitsandbytes 4bit：训练时低显存加载方案。
- MXFP4：gpt-oss 权重和推理后端相关的压缩形式。
- AWQ/GPTQ：离线推理量化格式。
- GGUF：本地推理文件格式。

## 3. bitsandbytes 4bit / 8bit

gpt-oss 如果走 QLoRA，可以使用 bitsandbytes 4bit 加载。但这和模型本身的 MXFP4 支持是两条线。

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

bnb_8bit_config = BitsAndBytesConfig(load_in_8bit=True)
print(bnb_4bit_config)
print(bnb_8bit_config)

## 4. GGUF / Ollama

如果只是本地运行 gpt-oss，优先使用官方 Ollama 路线：`ollama run gpt-oss:20b`。

自定义 GGUF 路线要确认转换工具是否支持 gpt-oss 的结构、Harmony 和相关权重格式。不能假设普通 dense 模型转换流程直接可用。

In [ ]:
print("ollama run gpt-oss:20b")

## 5. AWQ / GPTQ / EXL2 / FP8

这些路线能不能用于 gpt-oss，取决于当前工具链对 gpt-oss 架构、MoE、Harmony 和权重格式的支持。

写 notebook 时必须明确：

- 如果某个框架当前支持，给真实命令。
- 如果支持不完整，不要硬写假代码。
- 服务端优先看 vLLM 官方支持。
- 本地优先看 Ollama 官方模型。